In [ ]:
import subprocess
import librosa
import numpy as np
import json
import os
import time
import re
import urllib.parse as urlparse

timestamps_json = '''
'''

timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)


def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            # Adjust the output path to have an m4a extension
            output_path = output_path.replace('.ogg', '.m4a')
            
            command = [
                'yt-dlp', '-x', '--audio-format', 'm4a',
                '-o', output_path, youtube_url
            ]
            subprocess.run(command, check=True)
            return output_path
        except subprocess.CalledProcessError as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

def load_audio_segment(audio_path, start_time=0, end_time=None, sr=11025):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
        duration = end_total_seconds - start_total_seconds
    else:
        # If end_time is None, calculate the full length of the audio
        full_duration = librosa.get_duration(filename=audio_path)
        duration = full_duration - start_total_seconds

    y, sr = librosa.load(audio_path, sr=sr, offset=start_total_seconds, duration=duration)
    return y, sr

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=Bh_nHgbhYRg'
youtube_url2 = 'https://www.youtube.com/watch?v=Snq3QAtIk7A&t=38'

start_time1 = 1.88
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "44:28"
end_time2 = "42:32" #Use None for full length
full_duration2 = 0

# Download audio files
audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.ogg')
if audio_path1:
    print('audio_path1 downloaded')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.ogg')
if audio_path2:
    print('audio_path2 downloaded')

# Convert to OGG
ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
convert_to_ogg(audio_path1.replace('.ogg', '.m4a'), ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2.replace('.ogg', '.m4a'), ogg_path2, start_time2, end_time2)

In [ ]:
import gc
from concurrent.futures import ThreadPoolExecutor

HOP_LENGTH = 128


def process_chunk(y_chunk, sr, hop_length):
    # Chroma feature extraction
    chroma = librosa.feature.chroma_stft(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
    # chroma = normalize_features(chroma)
    
    # Constant-Q Transform feature extraction
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
    # CQT = normalize_features(CQT)
    
    # Ensure the same length for both arrays
    min_length = min(chroma.shape[1], CQT.shape[1])
    chroma = chroma[:, :min_length]
    CQT = CQT[:, :min_length]
    
    # Combine features
    combined_chunk_features = np.vstack((chroma, CQT)).T
    
    return combined_chunk_features

def load_and_preprocess_audio_combined(audio_path, target_sr=44100, chunk_duration=10, hop_length=HOP_LENGTH):
    y, sr = librosa.load(audio_path, sr=target_sr)
    y = normalize_audio(y)  # Normalize the raw audio signal
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = max(1, num_chunks // 10)  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")

    def process_and_collect(i):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        chunk_features = process_chunk(y_chunk, sr, HOP_LENGTH)
        return chunk_features

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_and_collect, i) for i in range(num_chunks)]
        for i, future in enumerate(futures):
            combined_features.append(future.result())
            # Print progress
            if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
                print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")
            gc.collect()

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

# Normalize the raw audio signal
def normalize_audio(y, epsilon=1e-8):
    return (y - np.mean(y)) / (np.std(y) + epsilon)
    
def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format

S1, sr1 = load_and_preprocess_audio_combined(ogg_path1)

In [ ]:
#making own step so no need to re-run audio1 for multiple recordings.

S2, sr2 = load_and_preprocess_audio_combined(ogg_path2)

In [ ]:
print(S1.shape,S2.shape)

In [ ]:
from fastdtw import fastdtw
#from dtaidistance import dtw
    
def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

#def constrained_dtw(S1, S2):
#    # Calculate the DTW distance with a window constraint
#    distance, paths = dtw.warping_paths(S1, S2, window=1000)
#    # Find the best path
#    path = dtw.best_path(paths)
#    return path

#warping_path = constrained_dtw(S1, S2)
warping_path = dynamic_time_warping_approx(S1, S2)

In [ ]:
def adjust_timestamps(wp, timestamps, sr):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t']) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1)

In [ ]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t'])
        ratios.append(current_ratio)
    return ratios

def format_number(number):
    return f"{number:.3f}"

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios)):  # Include the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios), i + neighbor_count + 1)  # Include the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.4 * neighborhood_average or adjusted_ratios[i] < 0.6 * neighborhood_average:
            duration = adjusted_timestamps[i + 1]['t'] - adjusted_timestamps[i]['t'] if i + 1 < len(adjusted_timestamps) else None
            if duration is not None:
                duration = format_number(duration)
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "duration": duration,
                "original_ratio": format_number(original_ratios[i]) if i < len(original_ratios) else "0.000",
                "adjusted_ratio": format_number(adjusted_ratios[i]),
                "average_neighbors": format_number(neighborhood_average)
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.40):
        difference = adjusted_t - previous_t
#        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")
print(youtube_url2)

def calculate_full_duration(audio_path, start_time, end_time):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
    else:
        # Calculate the full length of the audio if end_time is None
        full_duration = librosa.get_duration(path=audio_path)
        end_total_seconds = full_duration

    full_duration = end_total_seconds - start_total_seconds
    return full_duration

full_duration2 = calculate_full_duration(ogg_path2, start_time2, end_time2)

#Flag anything that exceeds 40% difference compared to neighboring measures.
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, full_duration2)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Duration: {ft['duration']}, Original: {ft['original_ratio']}, Neighbors' Avg.: {ft['average_neighbors']}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
os.remove(audio_path2)
#os.remove(ogg_path1)
os.remove(ogg_path2)